In [1]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

cp: cannot stat 'kaggle.json': No such file or directory


In [2]:
!kaggle datasets download -d jayeshrohansingh/emotion-detection-dataset

Dataset URL: https://www.kaggle.com/datasets/jayeshrohansingh/emotion-detection-dataset
License(s): other
100% 151M/151M [00:01<00:00, 110MB/s] 



In [3]:
import zipfile
zip_ref = zipfile.ZipFile('/content/emotion-detection-dataset.zip','r')
zip_ref.extractall('/content')
zip_ref.close()

In [ ]:
import tensorflow as tf
from tensorflow.keras  import utils
from tensorflow import keras
from keras import layers
from keras.layers import Conv2D,Dense,Flatten,MaxPooling2D
from tensorflow.keras.models import Sequential

In [ ]:
# loading training dataset
train_dataset=utils.image_dataset_from_directory(
    directory='/content/fer2013',
    labels='inferred',

    label_mode='int',
    batch_size=32,
    image_size=(48,48),
    seed=42,
    validation_split=0.2,
    subset='training')


validation_dataset=utils.image_dataset_from_directory(
    directory='/content/fer2013',
    labels='inferred',
    label_mode='int',
    batch_size=32,
    image_size=(48,48),
    validation_split=0.2,
    subset="validation",
    seed=42
)


Found 32298 files belonging to 2 classes.
Using 25839 files for training.
Found 32298 files belonging to 2 classes.
Using 6459 files for validation.


In [ ]:
from flax.nnx.nn import normalization
train_dataset=train_dataset.apply(tf.data.experimental.ignore_errors())
validation_dataset=validation_dataset.apply(tf.data.experimental.ignore_errors())
normalization_layers=tf.keras.layers.Rescaling(1./255)

Instructions for updating:
Use `tf.data.Dataset.ignore_errors` instead.


In [ ]:
model=Sequential()
# First convolutional layer
model.add(Conv2D(32,kernel_size=(3,3),activation='relu',input_shape=(48,48,3),padding='same'))
# MaxPooling layer / reducing image dimension
model.add(MaxPooling2D(pool_size=(2,2),strides=(2),padding='same'))


# second convolutional layer
model.add(Conv2D(64,kernel_size=(3,3),activation='relu',padding='same'))
model.add(MaxPooling2D(pool_size=(2,2),strides=(2),padding='same'))


# Converting 2D feature map into 1D array
model.add(Flatten())


# ANN layers
model.add(Dense(32,activation='relu'))
# Final layers
model.add(Dense(7,activation='softmax'))

In [ ]:
model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 48, 48, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 9216)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │       294,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           231 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 314,567 (1.20 MB)

 Trainable params: 314,567 (1.20 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy',metrics=['accuracy'])

In [ ]:
# Training the CNN model
prediction=model.fit(train_dataset,epochs=3,validation_data=validation_dataset)

Epoch 1/3
808/808 ━━━━━━━━━━━━━━━━━━━━ 89s 111ms/step - accuracy: 0.8899 - loss: 0.3554 - val_accuracy: 0.8854 - val_loss: 0.3693
Epoch 2/3
808/808 ━━━━━━━━━━━━━━━━━━━━ 88s 109ms/step - accuracy: 0.8899 - loss: 0.3516 - val_accuracy: 0.8851 - val_loss: 0.3634
Epoch 3/3
808/808 ━━━━━━━━━━━━━━━━━━━━ 89s 110ms/step - accuracy: 0.8900 - loss: 0.3475 - val_accuracy: 0.8853 - val_loss: 0.3639
